In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('stopwords')

# Load the dataset
df = pd.read_csv('/content/Laptop_Train_v2.csv')



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
df

,id,Sentence,Aspect Term,polarity,from,to
0,2339,I charge it at night and skip taking the cord ...,cord,neutral,41,45
1,2339,I charge it at night and skip taking the cord ...,battery life,positive,74,86
2,1316,The tech guy then said the service center does...,service center,negative,27,41
3,1316,The tech guy then said the service center does...,"""sales"" team",negative,109,121
4,1316,The tech guy then said the service center does...,tech guy,neutral,4,12
...,...,...,...,...,...,...
2353,2272,We also use Paralles so we can run virtual mac...,Windows Server Enterprise 2003,neutral,104,134
2354,2272,We also use Paralles so we can run virtual mac...,Windows Server 2008 Enterprise,neutral,140,170
2355,848,"How Toshiba handles the repair seems to vary, ...",repair,conflict,24,30
2356,848,"How Toshiba handles the repair seems to vary, ...",repair,positive,130,136


In [3]:
map_polarity = {'neutral': 2, 'positive': 1, "negative": 0} # map data

df["polarity"] = df["polarity"].map(map_polarity)

In [4]:
df=df.drop('from',axis=1)

In [5]:
df=df.drop('to',axis=1)

In [6]:
df

,id,Sentence,Aspect Term,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,2.0
1,2339,I charge it at night and skip taking the cord ...,battery life,1.0
2,1316,The tech guy then said the service center does...,service center,0.0
3,1316,The tech guy then said the service center does...,"""sales"" team",0.0
4,1316,The tech guy then said the service center does...,tech guy,2.0
...,...,...,...,...
2353,2272,We also use Paralles so we can run virtual mac...,Windows Server Enterprise 2003,2.0
2354,2272,We also use Paralles so we can run virtual mac...,Windows Server 2008 Enterprise,2.0
2355,848,"How Toshiba handles the repair seems to vary, ...",repair,NaN
2356,848,"How Toshiba handles the repair seems to vary, ...",repair,1.0


In [7]:
df.shape

(2358, 4)

In [8]:
df=df.dropna()

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2313 entries, 0 to 2357
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           2313 non-null   int64  
 1   Sentence     2313 non-null   object 
 2   Aspect Term  2313 non-null   object 
 3   polarity     2313 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 90.4+ KB


In [10]:
# Preprocess the data
def preprocess_text(Sentence):
    # Remove special characters and digits
    Sentence = re.sub('[^a-zA-Z]', ' ', Sentence)
    # Convert to lowercase
    Sentence = Sentence.lower()
    # Tokenize the text
    tokens = word_tokenize(Sentence)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    # Join the tokens back into a string
    Sentence = ' '.join(tokens)
    return Sentence


In [12]:
# Apply the preprocessing function to the text column
Sentence = df['Sentence'].apply(preprocess_text)


In [13]:
df.head()

,id,Sentence,Aspect Term,polarity
0,2339,charge night skip taking cord good battery life,cord,2.0
1,2339,charge night skip taking cord good battery life,battery life,1.0
2,1316,tech guy said service center exchange direct c...,service center,0.0
3,1316,tech guy said service center exchange direct c...,"""sales"" team",0.0
4,1316,tech guy said service center exchange direct c...,tech guy,2.0


In [14]:
Sentence

0         charge night skip taking cord good battery life
1         charge night skip taking cord good battery life
2       tech guy said service center exchange direct c...
3       tech guy said service center exchange direct c...
4       tech guy said service center exchange direct c...
                              ...                        
2352    also use paralles run virtual machines windows...
2353    also use paralles run virtual machines windows...
2354    also use paralles run virtual machines windows...
2356    toshiba handles repair seems vary folks indica...
2357    would like use different operating system alto...
Name: Sentence, Length: 2313, dtype: object

In [27]:
Sentence[5]

'high quality killer gui extremely stable highly expandable bundled lots good applications easy use absolutely gorgeous'

In [17]:
import tensorflow as tf
import tensorflow_hub as hub


In [20]:
#!python -m pip install tensorflow_text

In [21]:
import tensorflow_text as text

In [22]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(Sentence, df['polarity'], test_size=0.2, random_state=42)

In [25]:
def get_sentence_embeding(sentences):
    preprocessed_text = bert_preprocess(sentences)
    return bert_encoder(preprocessed_text)['pooled_output']

In [28]:
get_sentence_embeding([
    "high quality killer gui extremely stable highly expandable bundled lots good applications easy use absolutely gorgeous", 
    " charge night skip taking cord good battery life"]
)

<tf.Tensor: shape=(2, 768), dtype=float32, numpy=
array([[-0.8481645 , -0.43998545, -0.9007934 , ..., -0.78725225,
        -0.6722211 ,  0.87450635],
       [-0.9188357 , -0.37963182, -0.799814  , ..., -0.6524892 ,
        -0.62793183,  0.92653745]], dtype=float32)>